# Notebook: Supervisão e Segurança Lógica - Fábrica de Paçoca

Este notebook implementa o motor de inferência para as regras de alarme, intertravamento e segurança da planta de processamento. 
Utilizamos a lógica proposicional para validar as condições de operação segura e provar matematicamente a ausência de estados de risco.

In [ ]:
import itertools
import random
import pandas as pd

# Variáveis mapeadas no Setor 100, 200 e 400 (conforme Arquivo 02)
variaveis = [
    'u1', 'q1', 'e1', 'm1', # Setor 100
    't2', 'c1', 'v1', 'm2', 'a1', # Setor 200
    'p_air', 'i1', 'd1', 'v3' # Setor 400
]

# Gera todas as combinações possíveis (Exaustivo para prova lógica)
todas_as_variacoes = [
    dict(zip(variaveis, valores))
    for valores in itertools.product([False, True], repeat=len(variaveis))
]

def gerar_estado_aleatorio():
    return {v: random.choice([True, False]) for v in variaveis}

print(f'Total de combinações para teste: {len(todas_as_variacoes)}')

## 1. Definição das Expressões de Alarme e Intertravamento

### A. Não-Conformidade na Recepção
Bloqueia o motor da peneira ($m_1$) se houver excesso de umidade ($u_1$), acidez ($q_1$) ou emergência ($e_1$).

### B. Segurança Térmica do Forno
Fecha a válvula de gás ($v_1$) e ativa alarme ($a_1$) se houver chama ($c_1$) com esteira parada ($\neg m_2$) ou sobretemperatura ($t_2$).

### C. Sistema de Rejeição
Atua a válvula de descarte ($v_3$) se houver produto quebrado ($i_1$) ou metal ($d_1$), com pressão de ar ($p_{air}$) disponível.

In [ ]:
def alarme_recepcao(vars_):
    # F_qualidade -> not m1
    falha = vars_['u1'] or vars_['q1'] or vars_['e1']
    return (not falha) or (not vars_['m1'])

def intertrava_forno(vars_):
    # F_fogo -> (not v1 and a1)
    risco = (vars_['c1'] and not vars_['m2']) or vars_['t2']
    return (not risco) or (not vars_['v1'] and vars_['a1'])

def sistema_rejeito(vars_):
    # P_rejeito -> v3
    permissivo = (vars_['i1'] or vars_['d1']) and vars_['p_air']
    return (not permissivo) or vars_['v3']

def estado_risco_forno(vars_):
    # Situação proibida: Chama ativa, esteira parada e gás aberto
    return vars_['c1'] and (not vars_['m2']) and vars_['v1']

def regra_seguranca_forno(vars_):
    # (c1 and not m2) -> not v1
    return not (vars_['c1'] and not vars_['m2']) or (not vars_['v1'])

def prova_seguranca_matematica(vars_):
    # Prova que o risco e a regra de segurança juntos resultam em FALSO (Contradição)
    return estado_risco_forno(vars_) and regra_seguranca_forno(vars_)

## 2. Validação Formal (Tautologias e Contradições)

Abaixo avaliamos se as regras de segurança garantem que o sistema nunca entre em estado crítico.

In [ ]:
expressoes = {
    'Alarme Recepção (Regra)': alarme_recepcao,
    'Intertrava Forno (Regra)': intertrava_forno,
    'Sistema Rejeito (Regra)': sistema_rejeito,
    'Prova de Segurança do Forno (Contradição)': prova_seguranca_matematica
}

def classificar(fn):
    valores = [fn(v) for v in todas_as_variacoes]
    if all(valores): return 'Tautologia'
    if not any(valores): return 'Contradição'
    return 'Contingente'

resumos = []
for nome, fn in expressoes.items():
    valores = [fn(v) for v in todas_as_variacoes]
    resumos.append({
        'Expressão': nome,
        'Classificação': classificar(fn)
    })

print(pd.DataFrame(resumos).to_string(index=False))

### Conclusão do Diagnóstico
1. As **regras operacionais** são contingentes (dependem dos sensores).
2. A **Prova de Segurança** deve ser uma **Contradição**, provando que o estado de risco é impossível sob a lógica de intertravamento estabelecida.